# ML-08 — Model: Content Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring  
**Question:** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

This notebook compares a grouped-validation Random Forest model with the Week-4 readable baseline on the **same dataset, same held-out clients, and same ranking metric (Precision@K)**.

The target is `trend_direction == "down"`, used only as an evaluation label. It is **not** used as a feature. This is a decision-support experiment, not causal proof.


## 1) Method choice and why

I chose a **Random Forest classifier** because this lane contains mixed numeric signals and the relationship with a declining trend may be non-linear.

The model will be interpreted with feature importance. Complexity only counts as useful if it improves the same held-out ranking metric over the Week-4 baseline.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

# Evaluation label only; never supplied as a model feature.
df["target_declining"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df["target_declining"].value_counts(normalize=True).rename("share"))


## 2) Split design

A row-level split could put pages from the same client in both train and test. To avoid that, the split is **grouped by `client_id`**.

Approximately 80% of clients are used for training and 20% for testing. The test set is held out for the final comparison.


In [ ]:
candidate_features = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct"
]

features = [c for c in candidate_features if c in df.columns]
print("Features used:", len(features))
print(features)

X = (
    df[features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)
y = df["target_declining"].astype(int)
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

assert len(train_clients & test_clients) == 0


## 3) Train + compare vs my baseline

The Week-4 baseline ranked pages with the same two audited signals:

- 60% normalized staleness
- 40% log-normalized 90-day impressions

Both approaches are evaluated on the **same held-out test rows** using **Precision@K**. Higher Precision@K means more of the top-ranked review queue is labeled as declining.


In [ ]:
def minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

# Week-4 baseline on test rows only.
baseline_test = df.iloc[test_idx].copy()
baseline_test["staleness_norm"] = minmax(baseline_test["days_since_last_update"])
baseline_test["impressions_log"] = np.log1p(
    pd.to_numeric(baseline_test["impressions_90d"], errors="coerce").fillna(0)
)
baseline_test["impressions_norm"] = minmax(baseline_test["impressions_log"])
baseline_test["baseline_score"] = (
    0.60 * baseline_test["staleness_norm"]
    + 0.40 * baseline_test["impressions_norm"]
)

# Random Forest model.
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
model_prob = model.predict_proba(X_test)[:, 1]

baseline_score = baseline_test["baseline_score"].to_numpy()
y_true = y_test.to_numpy()

def precision_at_k(y_true, score, k):
    k = min(k, len(y_true))
    order = np.argsort(-score)[:k]
    return float(np.mean(y_true[order]))

rows = []
for k in [20, 50, 100]:
    base_p = precision_at_k(y_true, baseline_score, k)
    rf_p = precision_at_k(y_true, model_prob, k)
    rows.append({
        "K": k,
        "Week-4 baseline Precision@K": base_p,
        "Random Forest Precision@K": rf_p,
        "RF - baseline": rf_p - base_p
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

print("\nAdditional model metrics:")
print("ROC-AUC:", round(roc_auc_score(y_true, model_prob), 4))
print("Average Precision:", round(average_precision_score(y_true, model_prob), 4))


### Feature interpretation

Feature importance shows which variables helped the Random Forest separate the two evaluation classes in this split. It is **not causal evidence**.


In [ ]:
importance = (
    pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

print(importance.head(10).to_string(index=False))


## 4) Errors and interpretation

Inspect the model's top-ranked pages. The review distinguishes:

- **True positive:** a declining page prioritized near the top.
- **False positive:** a page prioritized near the top but not labeled declining.
- **False negative:** a declining page that the model ranked lower.

These are model-behavior observations, not causal explanations.


In [ ]:
review = df.iloc[test_idx][
    ["content_id", "client_id", "trend_direction",
     "days_since_last_update", "impressions_90d",
     "ctr", "avg_position", "engagement_rate"]
].copy()

review["actual_declining"] = y_true
review["model_score"] = model_prob
review["baseline_score"] = baseline_score

top20_model = review.sort_values("model_score", ascending=False).head(20).copy()
top20_model["error_type"] = np.where(
    top20_model["actual_declining"] == 1,
    "True positive",
    "False positive"
)

print(top20_model.to_string(index=False))

print("\nTop-20 false positives:",
      int((top20_model["actual_declining"] == 0).sum()))

print("Declining pages outside model top-20:",
      int(review.sort_values("model_score", ascending=False).iloc[20:]["actual_declining"].sum()))


In [ ]:
best_row = comparison_df.loc[comparison_df["RF - baseline"].idxmax()]

print("Model-vs-baseline conclusion:")
if best_row["RF - baseline"] > 0:
    print(
        f"Random Forest beats the Week-4 baseline most at K={int(best_row['K'])}, "
        f"with a Precision@K improvement of {best_row['RF - baseline']:.3f}."
    )
elif best_row["RF - baseline"] < 0:
    print(
        f"Random Forest does not beat the Week-4 baseline on this held-out split. "
        f"The baseline is stronger by {abs(best_row['RF - baseline']):.3f} at K={int(best_row['K'])}."
    )
else:
    print(f"Random Forest and baseline tie at K={int(best_row['K'])}.")

print("Use the result as decision-support evidence, not causal proof.")


## 5) Self-check

- Method choice explained.
- Client-grouped train/test split used.
- `trend_direction` and `trend_pct` are excluded from model features.
- Week-4 baseline reconstructed from the same two audited signals.
- Baseline and model evaluated on the same held-out rows.
- Same ranking metric used: Precision@K.
- ROC-AUC and Average Precision reported.
- Feature importance and top-ranked errors inspected.
- Complexity is not treated as success by itself.

**Limitation:** `trend_direction == "down"` is an observed dataset label. It does not establish causality. One grouped split is also not enough to claim production performance, so this should remain a ranking experiment for review prioritization.
